# Kateto — ORPO v7 (305 pares, 2xT4)

Corre el ORPO con el `_orpo_step` de `on_after_backward` sobre la base v6b.
En 4 GB locales el forward entraba pero el `ce` se iba a NaN; aca sobran 32 GB
y ctx 256, asi que si el NaN persiste es un bug de la implementacion, no de memoria.

In [ ]:
# CELL 1 — setup
import os, subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
subprocess.run([sys.executable,'-m','pip','install','-q','lightning','datasets','safetensors','transformers'],check=True)
subprocess.run(['git','clone','-q','https://github.com/Gonanf/kateto-train.git','/kaggle/working/kateto'],check=True)
os.chdir('/kaggle/working/kateto')
subprocess.run(['git','submodule','update','--init','--recursive'],check=True)
print(subprocess.run(['git','log','--oneline','-3'],capture_output=True,text=True).stdout)

In [ ]:
# CELL 2 — dataset ORPO desde /kaggle/input
import glob, json
cand = glob.glob('/kaggle/input/**/*orpo*.jsonl', recursive=True)
print('  candidatos:', cand)
assert cand, 'no encontre el dataset de ORPO en /kaggle/input'
src = cand[0]
rows = [json.loads(l) for l in open(src) if l.strip()]
print(f'  pares: {len(rows)}')
print('  claves:', list(rows[0].keys()))
os.makedirs('data', exist_ok=True)
open('data/kateto_orpo_v7.jsonl','w').write(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows))

In [ ]:
# CELL 3 — ORPO con on_after_backward, ctx 256
import subprocess, os
env = dict(os.environ)
env.update({
    'ORPO_DIAG':'1', 'SMOKE_STEPS':'305', 'SMOKE_ACCUM':'4',
    'SMOKE_LR':'5e-7', 'SMOKE_LR_FINAL':'1e-7', 'SMOKE_ORPO_LAMBDA':'0.05',
    'SMOKE_CTX':'256', 'SMOKE_PREC':'bf16', 'SMOKE_GRADCP':'0',
    'SMOKE_MERGE':'1', 'SMOKE_DETACH':'0', 'SMOKE_TRAINTYPE':'none',
    'SMOKE_DATA':'/kaggle/working/kateto/data/kateto_orpo_v7.jsonl',
    'BASE_MODEL': glob.glob('/kaggle/input/**/rwkv-0.pth', recursive=True)[0],
    'PYTHON_BIN': sys.executable,
    'REPO_DIR': '/kaggle/working/kateto',
    'SMOKE_OUT':'/kaggle/working/orpo-v7',
})
r = subprocess.run(['bash','scripts/smoke-orpo-1.5b-local.sh'], env=env,
                   capture_output=True, text=True)
print(r.stdout[-6000:])
print('STDERR:', r.stderr[-3000:])
print('rc =', r.returncode)

In [ ]:
# CELL 4 — veredicto
import re
log = r.stdout + r.stderr
na = len(re.findall(r'\[orpo\] paso \d+ ce=nan', log))
ok = len(re.findall(r'\[orpo\] paso \d+ ce=[0-9]', log))
print(f'  pasos sanos: {ok}   pasos con NaN: {na}')
print('  VEREDICTO:', 'LIMPIO' if na == 0 and ok > 0 else f'NaN en {na} pasos')
print(subprocess.run(['ls','-la','/kaggle/working/orpo-v7/proj-c256/'],capture_output=True,text=True).stdout)